# m04 lab · Sampling, filtering and the EP recording chain

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Between the tissue and the screen, an electrogram is filtered, sampled and quantized. Each step changes what you can
measure. You sample and alias sinusoids, measure quantization noise, build Butterworth and notch filters by hand and
check them, and pass the m03 unipolar electrogram through the bands used in one clinical mapping study to see what
happens to its shape and to its activation time.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Sampling: aliasing, and how the sampling rate limits activation timing | eq-m04-sampling |
| B | Quantization noise and the 6 dB-per-bit rule | eq-m04-quantization |
| C | Filters by hand: RC high-pass, Butterworth, checked against scipy | eq-m04-highpass, eq-m04-butterworth |
| D | Filtering a unipolar electrogram: shape, amplitude and timing | eq-m04-delay |
| E | Mains interference and the notch filter | eq-m04-notch |

Filter bands come from Delacretaz et al. (2001): minimally filtered unipolar 0.5–500 Hz, high-pass filtered unipolar
100 Hz, bipolar 30–500 Hz. The test signal is the m03 unipolar model in relative units. Each part ends with
**checks**. No downloads; an Octave copy of parts A–E is in `labs/octave/m04_lab.m`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import recording as R
rng = np.random.default_rng(4)

## Part A · Sampling

Sampling at $f_s$ cannot tell a sinusoid at $f$ from one at $|f - f_s\,\mathrm{round}(f/f_s)|$ (eq-m04-sampling), so
anything above $f_s/2$ must be removed before the converter. Then look at timing: the steepest downstroke of a
sampled unipolar electrogram can only be located to within about a sample unless you interpolate.

In [ ]:
fs = 1000.0
n = np.arange(100)
fa = R.alias_frequency(850.0, fs)
same = np.allclose(np.cos(2 * np.pi * 850 * n / fs), np.cos(2 * np.pi * fa * n / fs))
print(f"850 Hz sampled at 1 kHz looks like {fa:.0f} Hz: identical samples = {same}")

# activation-time error at 1 kHz: random sub-sample positions of the front
errs_raw, errs_fit = [], []
for k in range(400):
    t0 = rng.uniform(-0.5, 0.5)                       # ms, true activation
    t = np.arange(-50, 50, 1.0)                       # 1 kHz grid
    x = R.unipolar_egm(t - t0, c=0.5, w=0.3, h=1.0)   # slower front (0.5 m/s) so the downstroke spans a few samples
    d = np.gradient(x, t)
    errs_raw.append(t[np.argmin(d)] - t0)
    errs_fit.append(R.steepest_downstroke(t, x) - t0)
print(f"timing error at 1 kHz: nearest sample sd {np.std(errs_raw):.3f} ms (1/sqrt(12) ms = {1 / np.sqrt(12):.3f}), "
      f"with parabolic refinement sd {np.std(errs_fit):.3f} ms")

In [ ]:
# Checks
assert same and fa == 150.0
assert abs(np.std(errs_raw) - 1 / np.sqrt(12)) < 0.05
assert np.std(errs_fit) < 0.5 * np.std(errs_raw)

## Part B · Quantization

An $N$-bit converter spanning $\pm V_{FS}$ has a step (LSB) of $q = 2V_{FS}/2^N$. Rounding adds noise of standard
deviation $q/\sqrt{12}$, so a full-scale sine has a signal-to-quantization-noise ratio of $6.02N + 1.76$ dB
(eq-m04-quantization). The PhysioNet Intracardiac Atrial Fibrillation Database used in m03 and m31 was recorded at
1 kHz with 14-bit resolution.

In [ ]:
t = np.arange(200000)
x = 0.98 * np.sin(2 * np.pi * 0.01234567 * t)
rows = []
for bits in (10, 12, 14):
    xq, lsb = R.quantize(x, bits, 1.0)
    snr = 10 * np.log10(np.mean(x ** 2) / np.mean((xq - x) ** 2))
    rows.append((bits, np.std(xq - x) / (lsb / np.sqrt(12)), snr, float(R.sqnr_db(bits)) + 20 * np.log10(0.98)))
    print(f"{bits} bits: noise / (LSB/sqrt 12) = {rows[-1][1]:.3f}, SNR {snr:.2f} dB (formula {rows[-1][3]:.2f} dB)")
print(f"14-bit full-scale sine: {float(R.sqnr_db(R.IAFDB['bits'])):.1f} dB")

In [ ]:
# Checks
for bits, ratio, snr, formula in rows:
    assert abs(ratio - 1) < 0.02 and abs(snr - formula) < 0.1

A full-scale sine is the best case. A small electrogram that uses only a fraction of the range loses 6 dB for every
halving of its amplitude, which is why gain matters before the converter.

## Part C · Filters by hand

A first-order RC high-pass has $H(f) = (jf/f_c)/(1 + jf/f_c)$ (eq-m04-highpass): $1/\sqrt2$ and a 45° lead at $f_c$,
and far below $f_c$ it differentiates. A Butterworth filter of order $n$ has $|H|^2 = 1/(1 + (f/f_c)^{2n})$
(eq-m04-butterworth). Build first- and second-order digital Butterworth filters with the bilinear transform and check
them against scipy.

In [ ]:
H = R.rc_highpass(30.0, 30.0)
print(f"RC high-pass at fc: |H| = {abs(H):.4f}, phase {np.degrees(np.angle(H)):.1f} degrees")
worst = 0.0
for order in (1, 2):
    for kind in ("low", "high"):
        b, a = R.butter_digital(30.0, 1000.0, order, kind)
        bs, as_ = signal.butter(order, 30.0, btype=kind, fs=1000.0)
        worst = max(worst, np.max(np.abs(b - bs)), np.max(np.abs(a - as_)))
print(f"largest coefficient difference from scipy.signal.butter: {worst:.1e}")
f = np.logspace(-1, 3.3, 400)
for lo, hi in ((0.5, 500.0), (30.0, 500.0)):
    mag = R.butter_magnitude(f, lo, 2, "high") * R.butter_magnitude(f, hi, 2, "low")
    plt.loglog(f, mag, label=f"{lo}-{hi:.0f} Hz, second order")
plt.loglog(f, R.butter_magnitude(f, 100.0, 2, "high"), label="100 Hz high-pass")
plt.xlabel("frequency (Hz)"); plt.ylabel("|H|"); plt.legend(fontsize=8); plt.ylim(1e-3, 2); plt.show()

In [ ]:
# Checks
assert abs(abs(H) - 1 / np.sqrt(2)) < 1e-12 and abs(np.degrees(np.angle(H)) - 45) < 1e-9
assert worst < 1e-12

## Part D · Filtering a unipolar electrogram

Pass the m03 unipolar electrogram (front at 1 m/s, width 0.3 mm, electrode 1 mm away) through the three bands, causally
as a live recording system would and forward-and-backward as offline software can. A causal filter delays slow
features by its group delay, $1/(2\pi f_c)$ for a first-order low-pass and $\sqrt2/(2\pi f_c)$ for a second-order one
(eq-m04-delay); zero-phase filtering removes the delay but cannot run live.

In [ ]:
fs = 20000.0
t = np.arange(-100, 100, 1000 / fs)        # ms
x = R.unipolar_egm(t)
bands = {"0.5-500 Hz": (0.5, 500.0), "30-500 Hz": (30.0, 500.0), "100 Hz high-pass": (100.0, None)}
out = {}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for name, (hp, lp) in bands.items():
    for k, zp in enumerate((False, True)):
        y = R.chain(x, fs, hp=hp, lp=lp, zero_phase=zp)
        out[(name, zp)] = (R.steepest_downstroke(t, y), np.ptp(y[np.abs(t) < 20]) / np.ptp(x[np.abs(t) < 20]))
        ax[k].plot(t, y, label=name)
for k, title in enumerate(("causal", "zero-phase")):
    ax[k].plot(t, x, "k--", lw=1, label="unfiltered"); ax[k].set_xlim(-10, 10); ax[k].set_title(title); ax[k].set_xlabel("time (ms)")
ax[0].legend(fontsize=7); plt.show()
for (name, zp), (lat, ratio) in out.items():
    print(f"{name:17s} {'zero-phase' if zp else 'causal    '}: downstroke at {lat:+.3f} ms, peak-to-peak x{ratio:.2f}")

# a high-pass removes the slow tail: compare the signal 10 ms from activation with its peak
y100 = R.chain(x, fs, hp=100.0, zero_phase=True)
at10 = np.argmin(np.abs(t - 10.0))
tail_raw = abs(x[at10]) / np.max(np.abs(x))
tail_hp = abs(y100[at10]) / np.max(np.abs(y100))
print(f"10 ms after activation: unfiltered {100 * tail_raw:.1f}% of its peak, 100 Hz high-passed {100 * tail_hp:.2f}%")

# group delay of a causal low-pass on a slow bump
tb = np.arange(-0.2, 0.2, 1 / fs)
bump = np.exp(-(tb / 0.01) ** 2)
shift = tb[np.argmax(R.apply(*R.butter_digital(100.0, fs, 2, "low"), bump))] - tb[np.argmax(bump)]
print(f"second-order 100 Hz low-pass delays a 10-ms bump by {1000 * shift:.2f} ms (formula {1000 * R.lowpass_group_delay_dc(100.0, 2):.2f} ms)")

In [ ]:
# Checks
for name in bands:
    assert abs(out[(name, True)][0]) < 0.02                     # zero phase keeps the activation time
assert abs(out[("100 Hz high-pass", False)][0]) > 0.1           # a causal high-pass moves it
assert out[("100 Hz high-pass", True)][1] < out[("0.5-500 Hz", True)][1]
assert tail_hp < 0.1 * tail_raw
assert abs(shift / R.lowpass_group_delay_dc(100.0, 2) - 1) < 0.1

High-pass filtering removes the slow components of a unipolar signal, which is where far-field activity lives, and
keeps the sharp local deflection; it also changes the waveform's shape, which is why high-pass filtered unipolar
recordings limit what can be read from morphology (Stevenson and Soejima 2005). Causal filters also shift features in time; when a system annotates live, that shift is part of the
measurement.

## Part E · Mains interference and the notch

A second-order notch at $f_0$ with quality factor $Q$ removes $f_0$ completely and has a −3 dB bandwidth of $f_0/Q$
(eq-m04-notch). Add 50 Hz interference to the electrogram, notch it out, and measure what the notch does to the clean
signal.

In [ ]:
fs = 1000.0
t = np.arange(-3000, 3000, 1000 / fs)       # long enough for the notch's start-up transient to die away
clean = R.unipolar_egm(t, c=0.5)
noisy = clean + 0.2 * np.sin(2 * np.pi * 50 * t / 1000)
b, a = R.notch(50.0, 30.0, fs)
cleaned = R.apply(b, a, noisy, zero_phase=True)
mid = np.abs(t) < 300
resid_before = np.std((noisy - clean)[mid]); resid_after = np.std((cleaned - clean)[mid])
print(f"50 Hz interference: sd {resid_before:.3f} before, {resid_after:.4f} after the notch")
w, h = signal.freqz(b, a, worN=np.linspace(40, 60, 20001), fs=fs)
band = w[np.abs(h) < 1 / np.sqrt(2)]
print(f"-3 dB bandwidth {band.max() - band.min():.2f} Hz (f0 / Q = {50 / 30:.2f} Hz)")
distortion = np.max(np.abs(R.apply(b, a, clean, zero_phase=True) - clean)[mid]) / np.ptp(clean)
print(f"the notch changes the clean electrogram by at most {100 * distortion:.1f}% of its peak-to-peak")
plt.plot(t, noisy, color="0.7", label="with 50 Hz"); plt.plot(t, cleaned, label="notched"); plt.plot(t, clean, "k--", lw=1, label="clean")
plt.xlim(-30, 30); plt.xlabel("time (ms)"); plt.legend(fontsize=8); plt.show()

In [ ]:
# Checks
assert resid_after < 0.05 * resid_before
assert abs((band.max() - band.min()) / (50 / 30) - 1) < 0.02
assert distortion < 0.05

## Where to go next

- In part A, repeat the timing experiment at 2 kHz and at 500 Hz. How does the error scale?
- In part D, use first-order filters instead. Which band moves the activation time most?
- In part E, try $Q = 5$. What does a wider notch do to the electrogram?
- m05 analyzes the sampled signal in frequency and time; m22 annotates activation times from it.